# Volume 6, Chapter 1: Sets, Choice and the Real Numbers (Python companion)

Exercises first, solutions at the end. We list the rationals without repetition, code pairs and sequences by single integers, diagonalise against programs, compute exactly with real numbers given as Cauchy sequences, watch the algebraic numbers fill the line, and run the proof of Schröder–Bernstein. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★) Listing the rationals.** Generate the first $10^5$ terms of the Calkin–Wilf sequence with Newman's formula $q_{n+1}=1/(2\lfloor q_n\rfloor+1-q_n)$ (Exercise 1.9), using `fractions.Fraction`. Check that they are distinct and in lowest terms, and find the index of $\frac{355}{113}$ without generating the sequence up to it.

**P1.2 (★) Pairing functions.** Cantor's pairing $\pi(m,n)=\frac{(m+n)(m+n+1)}2+n$ is a bijection $\N^2\to\N$. Implement it and its inverse, check them on a large square, and use them to list $\N^3$ and all finite sequences of integers.

**P1.3 (★★) CS application: diagonalisation and the halting problem.** (a) Given any list of total functions $f_0,f_1,\ldots:\N\to\N$ (as Python functions), build one that is not in the list. (b) Suppose someone hands you `halts(prog, arg)`, a function claimed to decide whether `prog(arg)` terminates. Write the program `diag` that defeats it, and run the argument on a few candidate deciders.

**P1.4 (★★) CS application: exact real arithmetic.** Represent a real number $x$ as a function `x(n)` returning a rational within $2^{-n}$ of $x$, a Cauchy sequence with a known speed (Section 1.4). Implement sums, products, $\sqrt2$ and $e$, compute $\sqrt2\cdot\sqrt2-2$ to $100$ digits, and explain why no program can decide whether such an $x$ equals $0$.

**P1.5 (★) Algebraic numbers are countable.** List the integer polynomials by the height $h(P)=\deg P+\sum|a_i|$ of Exercise 1.2, collect their real roots up to height $h\le 9$, and draw how they cover $[-2,2]$. Which rationals and which quadratic irrationals appear first?

**P1.6 (★) Schröder–Bernstein by computer.** Take $A=B=\N$, $f(n)=2n$ and $g(n)=2n+1$. Compute the bijection $h$ of Theorem 1.15 and check on $\{0,\dots,10^4\}$ that it is injective and hits every element of a smaller window. Describe $h$ in words.

## Solutions

### P1.1

In [ ]:
from fractions import Fraction
from math import floor, gcd

def calkin_wilf(count):
    q = Fraction(1)
    for _ in range(count):
        yield q
        q = 1 / (2 * floor(q) + 1 - q)

seq = list(calkin_wilf(10**5))
print(seq[:15])
print("distinct:", len(set(seq)) == len(seq))
print("lowest terms:", all(gcd(q.numerator, q.denominator) == 1 for q in seq))

def index(a, b):
    """Walk up the tree from a/b: the path is Euclid's algorithm; left child = bit 0, right child = bit 1."""
    bits = []
    while (a, b) != (1, 1):
        if a < b:
            bits.append(0); b -= a
        else:
            bits.append(1); a -= b
    n = 1
    for bit in reversed(bits):
        n = 2 * n + bit
    return n

for a, b in [(1, 2), (3, 2), (2, 5), (355, 113)]:
    print(f"{a}/{b} is q_{index(a, b)}")
print("check:", seq[index(2, 5) - 1])

The sequence starts $1,\frac12,2,\frac13,\frac32,\frac23,3,\ldots$; every term is new and reduced. The index of $\frac{355}{113}$ is just below $2^{26}$: walking up the tree subtracts the smaller of $a,b$ from the larger, and $355/113=[3;7,16]$ needs $3+7+16-1=25$ subtractions before reaching $\frac11$, so the index has $26$ binary digits. The binary digits of the index are the run lengths of the continued fraction, a fact due to Stern and Brocot. Newman's formula is a *bijection* $\N^*\to\Q_{>0}$ computed in constant time per step: countability made algorithmic.

### P1.2

In [ ]:
from math import isqrt

def pair(m, n):
    return (m + n) * (m + n + 1) // 2 + n

def unpair(z):
    w = (isqrt(8 * z + 1) - 1) // 2      # the diagonal m + n = w containing z
    n = z - w * (w + 1) // 2
    return w - n, n

N = 300
values = {pair(m, n) for m in range(N) for n in range(N)}
print("injective on the square:", len(values) == N * N)
print("inverse:", all(unpair(pair(m, n)) == (m, n) for m in range(N) for n in range(N)))
print("first codes:", [unpair(z) for z in range(10)])

def triple(z):
    a, r = unpair(z)
    b, c = unpair(r)
    return a, b, c

def finite_sequence(z):
    """z = 0 is the empty sequence; otherwise z - 1 codes (length - 1, rest)."""
    if z == 0:
        return ()
    length, rest = unpair(z - 1)
    out = []
    for _ in range(length):
        x, rest = unpair(rest)
        out.append(x)
    return tuple(out) + (rest,)

print([triple(z) for z in range(6)])
print([finite_sequence(z) for z in range(12)])
seen = {finite_sequence(z) for z in range(20000)}
print("all sequences of length <= 3 with entries <= 4 appear:",
      all(s in seen for s in [(a,) for a in range(5)] + [(a, b) for a in range(5) for b in range(5)]))

$\pi$ walks the diagonals $m+n=w$ one after the other, which is exactly Cantor's picture of $\N\times\N$. Iterating it codes triples, and adding a length codes all finite sequences: the set of finite sequences of integers is countable, hence so is the set of all finite texts, all programs and all proofs. That single remark, together with the uncountability of $\R$, shows that most real numbers cannot be described by any program, the starting point of computability theory (Volume C3).

### P1.3

In [ ]:
# (a) Diagonal escape from any list of total functions
fs = [lambda n: 0, lambda n: n, lambda n: n * n, lambda n: 2 ** n, lambda n: n % 7, lambda n: 3 * n + 1]
d = lambda n: fs[n](n) + 1 if n < len(fs) else 0
print([(i, fs[i](i), d(i)) for i in range(len(fs))])   # d differs from f_i at i

# (b) Defeating any claimed halting decider
def make_diag(halts):
    def diag(prog):
        if halts(prog, prog):
            while True:      # loop forever
                pass
        return "halted"
    return diag

candidates = {
    "always yes": lambda prog, arg: True,
    "always no": lambda prog, arg: False,
    "yes if name is short": lambda prog, arg: len(getattr(prog, "__name__", "")) < 5,
}
for name, halts in candidates.items():
    diag = make_diag(halts)
    claim = halts(diag, diag)
    # what really happens: if claim is True, diag(diag) loops; if False, it halts
    truth = not claim
    print(f"{name:22s} claims diag(diag) halts: {claim!s:5s}  but in fact it {'halts' if truth else 'loops'}")

Whatever `halts` answers about `diag(diag)`, `diag` does the opposite, so every candidate decider is wrong on at least one input. This is Theorem 1.14 in disguise: programs play the role of $a$, "the set of inputs on which program $a$ halts" plays the role of $f(a)$, and `diag` is the diagonal set $D$. (We never run `diag(diag)` when it would loop; the table reports what it would do.)

### P1.4

In [ ]:
from fractions import Fraction as F

class Real:
    """x(n) returns a rational within 2^-n of x."""
    def __init__(self, approx):
        self.approx = approx
    def __call__(self, n):
        return self.approx(n)
    def __add__(self, other):
        return Real(lambda n: self(n + 1) + other(n + 1))
    def __sub__(self, other):
        return Real(lambda n: self(n + 1) - other(n + 1))
    def __mul__(self, other):
        def approx(n):
            # |xy - x'y'| <= |x||y - y'| + |y'||x - x'|; bound |x|, |y| by their 0-th approximation + 1
            bx = abs(self(0)) + 1
            by = abs(other(0)) + 2
            k = n + 2 + max(int(bx).bit_length(), int(by).bit_length())
            return self(k) * other(k)
        return Real(approx)

def sqrt2():
    def approx(n):
        lo, hi = F(1), F(2)
        while hi - lo > F(1, 2**n):          # bisection: exact and simple
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if mid * mid < 2 else (lo, mid)
        return lo
    return Real(approx)

def e():
    def approx(n):
        s, term, k = F(0), F(1), 0
        while term > F(1, 2**(n + 1)):      # tail after term k is < 2 * term(k+1)
            s += term; k += 1; term /= k
        return s
    return Real(approx)

r2 = sqrt2()
z = r2 * r2 - Real(lambda n: F(2))
from decimal import Decimal, getcontext
getcontext().prec = 95     # 2^-340 is about 4e-103: 95 digits are all correct
q = z(340)
print("sqrt2*sqrt2 - 2 is within 2^-340 of", float(q))
s = r2(340)
print("sqrt2 =", Decimal(s.numerator) / Decimal(s.denominator))
q = e()(340)
print("e     =", Decimal(q.numerator) / Decimal(q.denominator))

The class implements Section 1.4 with an explicit modulus of convergence, which is how exact real arithmetic libraries (iRRAM, CoRN, `mpmath`'s interval mode) work. Sums and products are computable, and every digit we print is guaranteed. Equality is not: to decide $x=0$ a program can only look at finitely many approximations, all of size $<2^{-n}$ for the $n$ it has examined, and the number $\sum_k 2^{-k}\mathbf 1[\text{program }P\text{ halts at step }k]$ is $0$ or not according to whether $P$ halts. A decider for $x=0$ would decide the halting problem of P1.3. So in exact arithmetic one can compute $\sqrt2\cdot\sqrt2-2$ to any precision, but never certify that it is $0$ without an algebraic argument.

### P1.5

In [ ]:
import itertools, numpy as np
import matplotlib.pyplot as plt

def polys_of_height(h):
    for d in range(1, h):
        budget = h - d
        for coeffs in itertools.product(range(-budget, budget + 1), repeat=d + 1):
            if coeffs[-1] != 0 and sum(map(abs, coeffs)) == budget:
                yield coeffs              # a_0, ..., a_d

roots_by_height = {}
seen = set()
for h in range(2, 10):
    found = []
    for c in polys_of_height(h):
        for r in np.roots(c[::-1]):
            if abs(r.imag) < 1e-9:
                x = round(r.real, 9)
                if x not in seen:
                    seen.add(x); found.append(x)
    roots_by_height[h] = sorted(found)
    print(h, len(found), "new real algebraic numbers; first:", found[:6])

fig, ax = plt.subplots(figsize=(9, 3))
for h, xs in roots_by_height.items():
    xs = [x for x in xs if -2 <= x <= 2]
    ax.scatter(xs, [h] * len(xs), s=4)
ax.set_xlabel("x"); ax.set_ylabel("height"); ax.set_title("Real algebraic numbers by height")
plt.show()

Height $2$ gives only $0$ (from $X$), height $3$ adds $\pm1$, height $4$ adds $\pm2$ and $\pm\frac12$, and height $5$ brings the first irrationals: $\pm\sqrt2$, $\pm\frac1{\sqrt2}$ and the four roots of $X^2\pm X-1$, the golden ratio and its relatives. Each height brings finitely many new numbers ($1,2,4,12,28,76,178,492$), which is the whole proof of countability; yet together they are dense in $\R$, since they contain $\Q$. A countable dense set can still be tiny: Chapter 6 shows that it has length $0$.

### P1.6

In [ ]:
LIMIT = 10**4
f = lambda n: 2 * n
g = lambda n: 2 * n + 1
g_image = {g(b) for b in range(LIMIT)}

# C = union of C_k, with C_0 = A \ g(B) and C_{k+1} = g(f(C_k))
C = set()
frontier = {a for a in range(LIMIT) if a not in g_image}
while frontier:
    C |= frontier
    frontier = {g(f(a)) for a in frontier if g(f(a)) < LIMIT} - C

def h(a):
    return f(a) if a in C else (a - 1) // 2

values = [h(a) for a in range(LIMIT)]
print("injective:", len(set(values)) == LIMIT)
print("hits all of 0..999:", set(range(1000)) <= set(values))
print([(a, h(a)) for a in range(12)])

Here $g(B)$ is the odd numbers, so $C_0$ is the even numbers and $C_{k+1}=4C_k+1$, which appends the binary digits $01$: $a\in C$ exactly when deleting all final blocks $01$ from its binary expansion leaves an even number. On $C$, $h$ doubles; elsewhere it deletes the final digit $1$. The proof is an algorithm: to compute $h(a)$, follow the chain backwards ($a\mapsto g^{-1}(a)\mapsto f^{-1}(\cdot)\mapsto\cdots$) and see where it stops.